# Building an LLM Agent for Math Problem Solving

This tutorial demonstrates how to build an LLM agent using Qwen models via Hugging Face Inference API to solve mathematical problems.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Environment Configuration](#env-config)
3. [Qwen API Client Setup](#api-setup)
4. [Basic Math Problem Solver](#basic-solver)
5. [Enhanced Agent with Step-by-Step Reasoning](#enhanced-agent)
6. [Advanced Features](#advanced-features)
7. [Examples and Demonstrations](#examples)

## Learning Objectives
- Understand how to interact with Qwen models via Hugging Face API
- Build a basic math problem-solving agent
- Implement step-by-step reasoning for complex problems
- Add error handling and model fallback mechanisms
- Create a reusable agent framework


## 1. Setup and Installation {#setup}

First, we need to install the required packages. 

**Option 1: Install from requirements.txt (recommended)**
```bash
pip install -r requirements.txt
```

**Option 2: Install directly in the notebook** (run the cell below)


In [ ]:
%%capture
%pip install -r requirements.txt


## 2. Environment Configuration {#env-config}

We'll use a `.env` file to securely store the Hugging Face token. 

**Important:** Create a `.env` file in the same directory as this notebook with the following content:
```
HF_TOKEN=your_huggingface_token_here
```

If you don't have a `.env` file yet, you can create one by running the cell below (but remember to replace the token with your actual token).


In [ ]:
import os
from dotenv import load_dotenv
import sys
import time

# Load environment variables from .env file
load_dotenv()

# Get HF_TOKEN from environment
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    print("ERROR: HF_TOKEN not found in .env file or environment variables!")
    print("Please create a .env file with: HF_TOKEN=your_token_here")
    sys.exit(1)
else:
    print("✓ HF_TOKEN loaded successfully from .env file")


## 3. Qwen API Client Setup {#api-setup}

Now let's set up the Qwen API client with fallback model support.


In [ ]:
from huggingface_hub import InferenceClient

# Version-agnostic import for HfHubHTTPError
try:
    # new(er) releases
    from huggingface_hub.errors import HfHubHTTPError
except Exception:
    try:
        # some older releases
        from huggingface_hub.utils._errors import HfHubHTTPError
    except Exception:
        # last resort: define a stub so the code still runs
        class HfHubHTTPError(Exception):
            pass

# Model configuration
PRIMARY_MODEL = "Qwen/Qwen2.5-72B-Instruct"  # Most advanced general Qwen model
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # Auto-fallback if 72B is gated/busy

def make_client(model_id: str) -> InferenceClient:
    """Create an InferenceClient for the specified model."""
    return InferenceClient(model=model_id, token=HF_TOKEN, timeout=120)

def chat_once(client: InferenceClient, messages, max_tokens: int = 1024, temperature: float = 0.2) -> str:
    """
    Non-streaming chat completion.
    
    Args:
        client: InferenceClient instance
        messages: List of message dictionaries with 'role' and 'content'
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature (lower = more deterministic)
    
    Returns:
        Generated response text
    """
    try:
        resp = client.chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return resp.choices[0].message["content"]
    except Exception as e:
        raise Exception(f"Error in chat_once: {str(e)}")

def chat_stream(client: InferenceClient, messages, max_tokens: int = 1024, temperature: float = 0.3):
    """
    Streaming chat completion (prints tokens as they arrive).
    """
    for event in client.chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,
    ):
        if event.choices and "delta" in event.choices[0]:
            delta = event.choices[0]["delta"].get("content", "")
            if delta:
                print(delta, end="", flush=True)
    print()  # newline at the end

def get_client_with_fallback():
    """
    Try to get a client with the primary model, fallback to secondary if needed.
    Returns: (client, model_id) tuple
    """
    for model_id in (PRIMARY_MODEL, FALLBACK_MODEL):
        try:
            client = make_client(model_id)
            # Test with a simple call
            test_messages = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'OK' if you can respond."}
            ]
            response = chat_once(client, test_messages, max_tokens=10)
            print(f"✓ Successfully connected to {model_id}")
            return client, model_id
        except HfHubHTTPError as e:
            print(f"⚠ HTTP error on {model_id}: {e}")
            if model_id == PRIMARY_MODEL:
                print("   Falling back to alternative model...")
                time.sleep(1)
            else:
                print("   All models failed. Check your token permissions.")
                raise
        except Exception as e:
            print(f"⚠ Unexpected error on {model_id}: {e}")
            if model_id == PRIMARY_MODEL:
                print("   Falling back to alternative model...")
                time.sleep(1)
            else:
                raise
    
    raise Exception("No working model available")

# Initialize client
print("Initializing Qwen API client...")
client, active_model = get_client_with_fallback()
print(f"Active model: {active_model}\n")


## 4. Basic Math Problem Solver {#basic-solver}

Let's start with a simple math problem solver. This is the foundation of our agent.


In [ ]:
class BasicMathSolver:
    """A basic math problem solver using Qwen LLM."""
    
    def __init__(self, client: InferenceClient, model_name: str):
        self.client = client
        self.model_name = model_name
        self.system_prompt = """You are an expert mathematician and problem solver. 
Your task is to solve mathematical problems accurately and provide clear solutions."""
    
    def solve(self, problem: str, show_reasoning: bool = False) -> str:
        """
        Solve a math problem.
        
        Args:
            problem: The math problem to solve (as a string)
            show_reasoning: If True, show step-by-step reasoning
        
        Returns:
            The solution to the problem
        """
        user_prompt = problem
        if show_reasoning:
            user_prompt += "\n\nPlease show your step-by-step reasoning."
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        response = chat_once(self.client, messages, max_tokens=512, temperature=0.1)
        return response
    
    def solve_stream(self, problem: str):
        """Solve a math problem with streaming output."""
        user_prompt = problem + "\n\nPlease show your step-by-step reasoning."
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        chat_stream(self.client, messages, max_tokens=512, temperature=0.1)

# Create an instance
basic_solver = BasicMathSolver(client, active_model)

# Test with a simple problem
print("Testing basic math solver:")
print("=" * 50)
problem1 = "What is 15 * 23 + 47?"
print(f"Problem: {problem1}\n")
solution1 = basic_solver.solve(problem1, show_reasoning=True)
print(f"Solution:\n{solution1}\n")


In [ ]:
class MathProblemAgent:
    """
    An enhanced LLM agent for solving math problems with step-by-step reasoning,
    error handling, and verification.
    """
    
    def __init__(self, client: InferenceClient, model_name: str):
        self.client = client
        self.model_name = model_name
        self.conversation_history = []
        
        self.system_prompt = """You are an expert mathematician and problem-solving agent. 
Your task is to solve mathematical problems with precision and clarity.

Guidelines:
1. Always show your step-by-step reasoning
2. Break down complex problems into smaller parts
3. Verify your calculations when possible
4. Clearly state your final answer
5. If you're uncertain, explain your reasoning process

Format your response as:
- Problem Analysis: [Your understanding of the problem]
- Step-by-Step Solution: [Detailed steps]
- Final Answer: [Clear answer]"""
    
    def solve(self, problem: str, verbose: bool = True) -> dict:
        """
        Solve a math problem with detailed reasoning.
        
        Args:
            problem: The math problem to solve
            verbose: If True, print the solution process
        
        Returns:
            Dictionary with problem, reasoning, answer, and metadata
        """
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": problem}
        ]
        
        if verbose:
            print(f"🤔 Problem: {problem}\n")
            print("🧠 Thinking...\n")
            print("-" * 60)
        
        try:
            response = chat_once(
                self.client, 
                messages, 
                max_tokens=2048, 
                temperature=0.1  # Low temperature for more deterministic math
            )
            
            # Store in conversation history
            self.conversation_history.append({
                "problem": problem,
                "response": response,
                "timestamp": time.time()
            })
            
            result = {
                "problem": problem,
                "reasoning": response,
                "model": self.model_name,
                "success": True
            }
            
            if verbose:
                print(response)
                print("-" * 60)
                print(f"✓ Solved using {self.model_name}\n")
            
            return result
            
        except Exception as e:
            error_msg = f"Error solving problem: {str(e)}"
            if verbose:
                print(f"❌ {error_msg}\n")
            
            return {
                "problem": problem,
                "error": error_msg,
                "success": False
            }
    
    def solve_stream(self, problem: str):
        """Solve a problem with streaming output for real-time feedback."""
        print(f"🤔 Problem: {problem}\n")
        print("🧠 Thinking...\n")
        print("-" * 60)
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": problem}
        ]
        
        chat_stream(self.client, messages, max_tokens=2048, temperature=0.1)
        print("-" * 60)
        print(f"✓ Solved using {self.model_name}\n")
    
    def solve_batch(self, problems: list, verbose: bool = True) -> list:
        """Solve multiple problems in sequence."""
        results = []
        for i, problem in enumerate(problems, 1):
            if verbose:
                print(f"\n[{i}/{len(problems)}]")
            result = self.solve(problem, verbose=verbose)
            results.append(result)
            if verbose and i < len(problems):
                print("\n" + "="*60 + "\n")
        return results
    
    def get_history(self) -> list:
        """Get conversation history."""
        return self.conversation_history

# Create an enhanced agent
agent = MathProblemAgent(client, active_model)


### Example 1: Simple Arithmetic


In [ ]:
result1 = agent.solve("Calculate: (125 + 347) * 23 - 156")


### Example 2: Algebra


In [ ]:
result2 = agent.solve("Solve for x: 3x + 7 = 2x - 5")


### Example 3: Word Problem


In [ ]:
result3 = agent.solve("""A train travels 240 miles in 3 hours. 
If it maintains the same speed, how long will it take to travel 400 miles?""")


### Example 4: Quadratic Equation


In [ ]:
result4 = agent.solve("Solve the quadratic equation: x² - 5x + 6 = 0")


### Example 5: Using Streaming Output


In [ ]:
agent.solve_stream("If a rectangle has a length of 12 cm and a width of 8 cm, what is its area and perimeter?")


## 6. Advanced Features {#advanced-features}

Let's add more advanced capabilities to our agent, including:
- Multi-step problem solving
- Problem verification
- Batch processing
- Conversation context


In [ ]:
class AdvancedMathAgent(MathProblemAgent):
    """
    An advanced math agent with additional features like verification,
    multi-step problem solving, and context awareness.
    """
    
    def __init__(self, client: InferenceClient, model_name: str):
        super().__init__(client, model_name)
        self.verification_prompt = """You are a math verification expert.
Given a problem and a proposed solution, verify if the solution is correct.
If incorrect, explain the error."""
    
    def solve_with_verification(self, problem: str, verify: bool = True) -> dict:
        """
        Solve a problem and optionally verify the answer.
        
        Args:
            problem: The math problem to solve
            verify: If True, verify the solution
        
        Returns:
            Dictionary with solution and verification
        """
        # First, solve the problem
        result = self.solve(problem, verbose=False)
        
        if not result["success"]:
            return result
        
        if verify:
            # Ask the model to verify its own solution
            verification_query = f"""Problem: {problem}
            
Proposed Solution:
{result['reasoning']}

Please verify if this solution is correct. Check:
1. Is the mathematical reasoning sound?
2. Are the calculations correct?
3. Is the final answer correct?

Provide a brief verification."""
            
            verify_messages = [
                {"role": "system", "content": self.verification_prompt},
                {"role": "user", "content": verification_query}
            ]
            
            try:
                verification = chat_once(
                    self.client,
                    verify_messages,
                    max_tokens=512,
                    temperature=0.1
                )
                result["verification"] = verification
                
                print(f"🤔 Problem: {problem}\n")
                print("=" * 60)
                print(result["reasoning"])
                print("=" * 60)
                print("\n🔍 Verification:")
                print(verification)
                print("=" * 60 + "\n")
                
            except Exception as e:
                result["verification_error"] = str(e)
        
        return result
    
    def solve_with_context(self, problem: str, context: str = None) -> dict:
        """
        Solve a problem with additional context.
        
        Args:
            problem: The math problem to solve
            context: Additional context or constraints
        """
        full_problem = problem
        if context:
            full_problem = f"Context: {context}\n\nProblem: {problem}"
        
        return self.solve(full_problem)
    
    def solve_multi_step(self, problem: str, max_steps: int = 5) -> dict:
        """
        Solve a complex problem by breaking it into multiple steps.
        Useful for very complex problems that might need iterative refinement.
        """
        print(f"🤔 Complex Problem: {problem}\n")
        print(f"🔀 Breaking into {max_steps} steps...\n")
        
        # First, ask the model to break down the problem
        breakdown_prompt = f"""Break down this problem into {max_steps} clear steps:
{problem}

Provide a numbered list of steps."""
        
        breakdown_messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": breakdown_prompt}
        ]
        
        try:
            breakdown = chat_once(self.client, breakdown_messages, max_tokens=1024)
            print("📋 Problem Breakdown:")
            print(breakdown)
            print("\n" + "=" * 60 + "\n")
            
            # Now solve the full problem
            return self.solve(problem)
            
        except Exception as e:
            return {
                "problem": problem,
                "error": f"Error in multi-step solving: {str(e)}",
                "success": False
            }

# Create an advanced agent
advanced_agent = AdvancedMathAgent(client, active_model)


### Example: Solving with Verification


In [ ]:
advanced_agent.solve_with_verification(
    "A store sells apples for $2.50 per pound and oranges for $3.00 per pound. "
    "If you buy 3 pounds of apples and 2 pounds of oranges, how much will you pay in total?",
    verify=True
)


### Example: Multi-Step Problem Solving


In [ ]:
complex_problem = """A company's revenue increased by 20% in the first quarter, 
then decreased by 15% in the second quarter. If the original revenue was $100,000, 
what is the revenue after both quarters? Also, calculate the overall percentage change."""
advanced_agent.solve_multi_step(complex_problem, max_steps=4)


### Example: Batch Processing


In [ ]:
batch_problems = [
    "What is 15% of 240?",
    "Solve for x: 2x - 8 = 12",
    "Calculate the area of a circle with radius 5 cm (use π ≈ 3.14)"
]

results = agent.solve_batch(batch_problems, verbose=True)


## 7. Examples and Demonstrations {#examples}

Let's test our agent with various types of math problems.


### Example Set 1: Basic Operations


In [ ]:
test_problems = [
    "What is 456 ÷ 12?",
    "Calculate: 7² + 3³",
    "Find the square root of 144"
]

for problem in test_problems:
    agent.solve(problem)
    print("\n" + "="*60 + "\n")


### Example Set 2: Word Problems


In [ ]:
word_problems = [
    """Sarah has 3 times as many books as John. If John has 12 books, 
    how many books do they have together?""",
    
    """A car travels 180 miles in 3 hours. What is its average speed in miles per hour?""",
    
    """A recipe calls for 2.5 cups of flour for every 1 cup of sugar. 
    If you use 4 cups of sugar, how many cups of flour do you need?"""
]

for problem in word_problems:
    agent.solve(problem)
    print("\n" + "="*60 + "\n")


### Example Set 3: Geometry Problems


In [ ]:
geometry_problems = [
    "Find the perimeter of a rectangle with length 15 cm and width 8 cm.",
    "Calculate the area of a triangle with base 10 cm and height 6 cm.",
    "What is the volume of a cube with side length 5 cm?"
]

for problem in geometry_problems:
    agent.solve(problem)
    print("\n" + "="*60 + "\n")


### Example Set 4: Advanced Problems


In [ ]:
advanced_problems = [
    "Solve the system of equations: 2x + 3y = 12 and x - y = 1",
    "Find the derivative of f(x) = 3x² + 2x - 5",
    "Calculate the sum of the first 10 terms of the arithmetic sequence: 5, 9, 13, 17, ..."
]

for problem in advanced_problems:
    agent.solve(problem)
    print("\n" + "="*60 + "\n")


## Summary

In this tutorial, we've learned:

1. **Setup**: How to configure Hugging Face API with Qwen models using environment variables
2. **Basic Agent**: Created a simple math problem solver
3. **Enhanced Agent**: Built an agent with step-by-step reasoning capabilities
4. **Advanced Features**: Added verification, multi-step solving, and batch processing
5. **Practical Examples**: Demonstrated the agent on various math problem types

## Key Takeaways

- **Error Handling**: Implement fallback mechanisms for API reliability
- **Temperature Settings**: Use lower temperatures (0.1-0.2) for mathematical accuracy
- **Step-by-Step Reasoning**: Prompt engineering is crucial for getting detailed solutions
- **Streaming**: Useful for real-time feedback and better user experience

## Next Steps

You can extend this agent by:
- Adding symbolic math libraries (SymPy) for verification
- Implementing a calculator tool that the LLM can call
- Building a web interface for the agent
- Adding support for more complex mathematical domains (calculus, statistics, etc.)
- Creating a conversation memory system for follow-up questions

## Resources

- [Hugging Face Inference API Documentation](https://huggingface.co/docs/api-inference/index)
- [Qwen Models on Hugging Face](https://huggingface.co/Qwen)
- [Python-dotenv Documentation](https://pypi.org/project/python-dotenv/)


## Bonus: View Conversation History

You can view all the problems solved in this session:


In [ ]:
history = agent.get_history()
print(f"Total problems solved: {len(history)}\n")
print("=" * 60)
for i, entry in enumerate(history, 1):
    print(f"\n[{i}] Problem: {entry['problem']}")
    print(f"    Solved at: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(entry['timestamp']))}")
    print("-" * 60)
